# 04 — Exact automatic differentiation

**Objectives.** Form gradients, Jacobians, and Hessians; understand forward/reverse products; verify derivatives numerically.

For $f:\mathbb R^n\to\mathbb R^m$, the Jacobian is $J_{ij}=\partial f_i/\partial x_j$. Forward mode computes $Jv$ efficiently; reverse mode computes $w^T J$. Hessians describe scalar-objective curvature and appear in Newton/KKT systems.


In [ ]:
import casadi as ca
import numpy as np
x = ca.MX.sym('x', 3)
f = ca.vertcat(x[0]*x[1], ca.sin(x[1]) + x[2]**2)
J = ca.jacobian(f, x)
phi = ca.dot(f, f)
H, grad = ca.hessian(phi, x)
derivatives = ca.Function('derivatives', [x], [f, J, grad, H])
values = derivatives([1.0, 0.4, -0.2])
print('J =\n', values[1], '\nH =\n', values[3])


In [ ]:
x0 = np.array([1.0, 0.4, -0.2]); direction = np.array([0.3, -0.1, 0.2]); eps = 1e-6
F = ca.Function('F', [x], [f])
finite_difference = (F(x0 + eps*direction) - F(x0 - eps*direction))/(2*eps)
exact = values[1] @ direction
np.testing.assert_allclose(finite_difference.full().ravel(), exact.full().ravel(), rtol=1e-6, atol=1e-8)


**Common mistake:** finite differences can validate a smooth graph, but step-size cancellation makes them a poor replacement for AD. Nondifferentiable `fabs`, `fmax`, and conditionals require modeling judgment. **Recap:** choose derivatives by the product needed, and inspect their sparsity.
